# Statistics and Evaluation

> **Note (API-free version):** Wherever the original notebook called the Gemini API, this version **prints the full prompt** to the console instead. You copy that prompt into any AI chat (ChatGPT, Claude, Gemini, etc.), paste the JSON response back, and the evaluation continues automatically.
>
> Cells that relied on `requests` / API keys have been removed. All local computation (sentence-transformers, BPMN parsing, Excel export) is unchanged.

---
## Shared helper: multi-line manual LLM input

Run this cell once. The function `manual_llm_query()` replaces every API call throughout the notebook. It prints a clearly delimited prompt, then waits for you to paste the AI's JSON response and type `###END###` on its own line.

In [1]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def manual_llm_query(prompt: str, label: str = "") -> str:
    """
    Replaces the Gemini API call.

    Prints the prompt so you can paste it into any AI chat, then collects
    the response interactively.  Type  ###END###  on its own line to finish.

    Args:
        prompt: The full prompt text to send to the AI.
        label:  Optional short description shown in the header (e.g. file name).

    Returns:
        The raw text the user pasted as the AI response.
    """
    header = f"PROMPT{'  [' + label + ']' if label else ''}"
    separator = "=" * 80
    print(f"\n{separator}")
    print(f"  {header}")
    print(separator)
    print(prompt)
    print(separator)
    print("  ↑  Copy everything between the lines above into your AI chat.")
    print("  Then paste the AI's JSON response below.")
    print("  When done, type  ###END###  on its own line and press Enter.")
    print(separator)

    lines = []
    while True:
        try:
            line = input()
        except EOFError:
            break
        if line.strip() == "###END###":
            break
        lines.append(line)

    response = "\n".join(lines)
    print(f"  ✓  Response received ({len(response)} chars).\n")
    return response


def manual_llm_layout_query(image_path: str) -> str:
    """
    Replaces the Gemini Vision API call for layout analysis.

    Prints instructions on how to upload the image plus the full evaluation
    prompt, then collects the AI's JSON response interactively.

    Args:
        image_path: Local path to the BPMN diagram image (.png / .jpg).

    Returns:
        The raw JSON text the user pasted as the AI response.
    """
    prompt_text = """\
You are a meticulous BPMN 2.0 layout quality analyst. Your task is to analyze \
the provided BPMN diagram image and identify all layout issues.

**Analysis Criteria:**
Please check for all three of the following categories of issues:
1. **Text Overlaps:** Any text label that overlaps with another label or a BPMN
   element's (e.g. pools) boundary.
2. **Element Overlaps:** Any BPMN shapes (tasks, events, gateways) that overlap
   each other. A task must be fully contained within its lane. Sequence flows
   (solid lines) must not cross over tasks or other elements. Note: Message flows
   (dashed lines) are permitted to cross over pools.
3. **Non-Orthogonal Edges:** Any sequence flow (solid line with a solid arrowhead)
   that is not horizontal or vertical. Diagonal or curved sequence flows are
   considered layout issues.

**Output Format:**
Respond with a single JSON object only. Do not include any other text or markdown
formatting.
The JSON object should have two keys:
- "overall_score": A single float from 0.0 (very poor) to 1.0 (perfect),
  representing the overall layout quality.
- "identified_issues": A list of objects. Each object must have two keys:
  "issue_description" (a clear, concise description of a single layout problem)
  and "severity" (a string: "High", "Medium", or "Low"). If there are no issues,
  return an empty list.

**Example Response:**
{
  "overall_score": 0.65,
  "identified_issues": [
    {
      "issue_description": "The label for 'Validate Order' task overlaps with the task boundary.",
      "severity": "Medium"
    },
    {
      "issue_description": "The sequence flow from 'Approve Order' to 'Ship Order' is drawn diagonally.",
      "severity": "High"
    }
  ]
}"""

    separator = "=" * 80
    print(f"\n{separator}")
    print(f"  LAYOUT ANALYSIS PROMPT  [{image_path}]")
    print(separator)
    print(f"  1. Open your AI chat that supports image uploads (e.g. ChatGPT, Claude, Gemini).")
    print(f"  2. Upload this image file:  {image_path}")
    print(f"  3. Copy and paste the prompt below into the chat (together with the image).")
    print(separator)
    print(prompt_text)
    print(separator)
    print("  Paste the AI's JSON response below.")
    print("  When done, type  ###END###  on its own line and press Enter.")
    print(separator)

    lines = []
    while True:
        try:
            line = input()
        except EOFError:
            break
        if line.strip() == "###END###":
            break
        lines.append(line)

    response = "\n".join(lines)
    print(f"  ✓  Response received ({len(response)} chars).\n")
    return response


print("Helper functions loaded: manual_llm_query(), manual_llm_layout_query()")

Helper functions loaded: manual_llm_query(), manual_llm_layout_query()


---
# I. Input Statistics

### Text statistics

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
text = """
Article 48

Request for a European cybersecurity certification scheme

1.   The Commission may request ENISA to prepare a candidate scheme or to review an existing European cybersecurity certification scheme on the basis of the Union rolling work programme.
2.   In duly justified cases, the Commission or the ECCG may request ENISA to prepare a candidate scheme or to review an existing European cybersecurity certification scheme which is not included in the Union rolling work programme. The Union rolling work programme shall be updated accordingly.
Article 49

Preparation, adoption and review of a European cybersecurity certification scheme

1.   Following a request from the Commission pursuant to Article 48, ENISA shall prepare a candidate scheme which meets the requirements set out in Articles 51, 52 and 54.
2.   Following a request from the ECCG pursuant to Article 48(2), ENISA may prepare a candidate scheme which meets the requirements set out in Articles 51, 52 and 54. If ENISA refuses such a request, it shall give reasons for its refusal. Any decision to refuse such a request shall be taken by the Management Board.
3.   When preparing a candidate scheme, ENISA shall consult all relevant stakeholders by means of a formal, open, transparent and inclusive consultation process.
4.   For each candidate scheme, ENISA shall establish an ad hoc working group in accordance with Article 20(4) for the purpose of providing ENISA with specific advice and expertise.
5.   ENISA shall closely cooperate with the ECCG. The ECCG shall provide ENISA with assistance and expert advice in relation to the preparation of the candidate scheme and shall adopt an opinion on the candidate scheme.
6.   ENISA shall take utmost account of the opinion of the ECCG before transmitting the candidate scheme prepared in accordance with paragraphs 3, 4 and 5 to the Commission. The opinion of the ECCG shall not bind ENISA, nor shall the absence of such an opinion prevent ENISA from transmitting the candidate scheme to the Commission.
7.   The Commission, based on the candidate scheme prepared by ENISA, may adopt implementing acts providing for a European cybersecurity certification scheme for ICT products, ICT services and ICT processes which meets the requirements set out in Articles 51, 52 and 54. Those implementing acts shall be adopted in accordance with the examination procedure referred to in Article 66(2).
8.   At least every five years, ENISA shall evaluate each adopted European cybersecurity certification scheme, taking into account the feedback received from interested parties. If necessary, the Commission or the ECCG may request ENISA to start the process of developing a revised candidate scheme in accordance with Article 48 and this Article.
"""

# Tokenize the text
tokens = nltk.word_tokenize(text)

# Count the total number of tokens
total_tokens = len(tokens)

# Split the text into sentences and count tokens for each sentence
sentences = nltk.sent_tokenize(text)
sentence_token_counts = [len(nltk.word_tokenize(sentence)) for sentence in sentences]

# Calculate average sentence length (in tokens)
average_sentence_length_tokens = total_tokens / len(sentences)

print(f"Total tokens: {total_tokens}")
print(f"Total sentences: {len(sentences)}")
print(f"Average sentence length (tokens): {average_sentence_length_tokens}")
print(f"Sentence token counts: {sentence_token_counts}")

Total tokens: 492
Total sentences: 27
Average sentence length (tokens): 18.22222222222222
Sentence token counts: [11, 29, 2, 38, 10, 15, 31, 2, 34, 15, 15, 2, 26, 2, 33, 2, 8, 29, 2, 30, 29, 2, 46, 21, 2, 26, 30]


### Model statistics

In [ ]:
import xml.etree.ElementTree as ET
import os

def analyze_bpmn(file_path):
    """
    Parses a .bpmn file to count nodes (activities and events),
    gateways, edges (sequence flows), and roles (lanes).
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at '{file_path}'")
        return None

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        namespace = {'bpmn': root.tag.split('}')[0][1:]}

        activity_types = [
            'task', 'sendTask', 'receiveTask', 'userTask', 'manualTask',
            'businessRuleTask', 'serviceTask', 'scriptTask', 'callActivity', 'subProcess'
        ]
        event_types = [
            'startEvent', 'endEvent', 'intermediateThrowEvent',
            'intermediateCatchEvent', 'boundaryEvent'
        ]
        gateway_types = [
            'exclusiveGateway', 'parallelGateway', 'inclusiveGateway',
            'eventBasedGateway', 'complexGateway'
        ]
        edge_types = ['sequenceFlow']
        role_types = ['pool', 'lane']

        total_activities = sum(len(root.findall(f".//bpmn:{t}", namespace)) for t in activity_types)
        total_events = sum(len(root.findall(f".//bpmn:{t}", namespace)) for t in event_types)
        total_nodes = total_activities + total_events
        total_gateways = sum(len(root.findall(f".//bpmn:{t}", namespace)) for t in gateway_types)
        total_edges = sum(len(root.findall(f".//bpmn:{t}", namespace)) for t in edge_types)
        total_roles = sum(len(root.findall(f".//bpmn:{t}", namespace)) for t in role_types)

        if total_roles == 0:
            total_roles = len(root.findall('.//bpmn:participant', namespace))

        counts = {
            "Total Nodes (Activities + Events)": total_nodes,
            "Total Gateways": total_gateways,
            "Total Edges (Sequence Flows)": total_edges,
            "Total Roles (Lanes/Participants)": total_roles
        }
        return counts

    except ET.ParseError:
        print(f"Error: Could not parse '{file_path}'. Make sure it is a valid BPMN/XML file.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


file_path = "/content/1_AI_Act.bpmn"
analysis_results = analyze_bpmn(file_path)

if analysis_results:
    print("\n--- BPMN Analysis Results ---")
    for key, value in analysis_results.items():
        print(f"{key}: {value}")
    print("---------------------------\n")


--- BPMN Analysis Results ---
Total Nodes (Activities + Events): 13
Total Gateways: 0
Total Edges (Sequence Flows): 11
Total Roles (Lanes/Participants): 2
---------------------------



---
# II. Semantic EVAL

### 1. Completeness

In [2]:
!pip install sentence-transformers torch pandas openpyxl

In [3]:
# ── Completeness check (no API needed) ───────────────────────────────────────
import xml.etree.ElementTree as ET
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import os

NAMESPACES = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}

def get_elements_by_tag(root, tag_name):
    return root.findall(f'.//bpmn:{tag_name}', NAMESPACES)

def get_element_names(elements):
    names = [elem.get('name') for elem in elements]
    return [name for name in names if name]

def get_actors(root):
    participants = get_elements_by_tag(root, 'participant')
    lanes = get_elements_by_tag(root, 'lane')
    return get_element_names(participants + lanes)

def get_activities(root):
    activity_tags = ['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask',
                     'manualTask', 'businessRuleTask', 'scriptTask', 'callActivity', 'subProcess']
    activities = []
    for tag in activity_tags:
        activities.extend(get_elements_by_tag(root, tag))
    return get_element_names(activities)

def get_events(root):
    event_tags = ['startEvent', 'endEvent', 'intermediateThrowEvent', 'intermediateCatchEvent', 'boundaryEvent']
    events = []
    for tag in event_tags:
        events.extend(get_elements_by_tag(root, tag))
    return get_element_names(events)

def get_gateways_count(root):
    and_gateways = get_elements_by_tag(root, 'parallelGateway')
    xor_gateways = get_elements_by_tag(root, 'exclusiveGateway')
    return {'AND': len(and_gateways), 'XOR': len(xor_gateways)}

def get_data_objects(root):
    data_objects = get_elements_by_tag(root, 'dataObjectReference')
    return get_element_names(data_objects)

def get_conditions(root):
    conditions = []
    xor_gateways = root.findall('.//bpmn:exclusiveGateway', NAMESPACES)
    for gateway in xor_gateways:
        gateway_id = gateway.get('id')
        outgoing_flows = [
            flow for flow in root.findall('.//bpmn:sequenceFlow', NAMESPACES)
            if flow.get('sourceRef') == gateway_id
        ]
        if len(outgoing_flows) > 1:
            condition_text = gateway.get('name')
            if condition_text:
                conditions.append(condition_text)
    return conditions


def calculate_recall_for_semantic_elements(gold_standard_names, generated_names, similarity_threshold, model):
    if not gold_standard_names:
        return 1.0, 0, 0, [], [], generated_names
    if not generated_names:
        return 0.0, 0, len(gold_standard_names), [], gold_standard_names, []

    gold_embeddings = model.encode(gold_standard_names, convert_to_tensor=True)
    generated_embeddings = model.encode(generated_names, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(gold_embeddings, generated_embeddings)

    matches_list = []
    unmatched_gold_indices = set(range(len(gold_standard_names)))
    unmatched_generated_indices = set(range(len(generated_names)))
    matched_generated_indices = set()

    for gs_idx in range(len(gold_standard_names)):
        best_score = -1
        best_gen_idx = -1
        for gen_idx in range(len(generated_names)):
            if gen_idx in matched_generated_indices:
                continue
            score = cosine_scores[gs_idx][gen_idx]
            if score > best_score:
                best_score = score
                best_gen_idx = gen_idx

        if best_score >= similarity_threshold:
            matches_list.append(
                (gold_standard_names[gs_idx], generated_names[best_gen_idx], best_score.item())
            )
            if best_gen_idx != -1:
                matched_generated_indices.add(best_gen_idx)
            if gs_idx in unmatched_gold_indices:
                unmatched_gold_indices.remove(gs_idx)
            if best_gen_idx in unmatched_generated_indices:
                unmatched_generated_indices.remove(best_gen_idx)

    matches_count = len(matches_list)
    total_gold = len(gold_standard_names)
    recall = matches_count / total_gold if total_gold > 0 else 1.0
    unmatched_gold = [gold_standard_names[i] for i in unmatched_gold_indices]
    unmatched_generated = [generated_names[i] for i in unmatched_generated_indices]

    return recall, matches_count, total_gold, matches_list, unmatched_gold, unmatched_generated


def calculate_recall_for_counts(gold_standard_count, generated_count):
    if gold_standard_count == 0:
        return 1.0, 0, 0
    matches = min(gold_standard_count, generated_count)
    recall = matches / gold_standard_count
    return recall, matches, gold_standard_count

def format_matches_for_excel(matches_list, unmatched_gold, unmatched_gen):
    report_parts = []
    if matches_list:
        report_parts.append("MATCHED:")
        for gs, gen, score in sorted(matches_list, key=lambda x: x[2], reverse=True):
            report_parts.append(f"  - G: '{gs}' <-> M: '{gen}' ({score:.2f})")
    if unmatched_gold:
        report_parts.append("\nMISSING FROM MODEL:")
        for name in sorted(unmatched_gold):
            report_parts.append(f"  - {name}")
    if unmatched_gen:
        report_parts.append("\nEXTRA IN MODEL:")
        for name in sorted(unmatched_gen):
            report_parts.append(f"  - {name}")
    return "\n".join(report_parts)


def run_completeness_check(gold_standard_file, generated_file, run_folder, thresholds, model):
    results = {
        'Gold Standard': os.path.basename(gold_standard_file),
        'Run Folder': run_folder,
        'Generated Model': os.path.basename(generated_file),
    }

    try:
        gold_root = ET.parse(gold_standard_file).getroot()
    except (FileNotFoundError, ET.ParseError) as e:
        print(f"Error reading gold standard file '{gold_standard_file}': {e}")
        return None

    try:
        generated_root = ET.parse(generated_file).getroot()
    except (FileNotFoundError, ET.ParseError) as e:
        print(f"Error reading generated file '{generated_file}': {e}")
        return None

    all_elements = {
        'Actors':       (get_actors(gold_root),       get_actors(generated_root)),
        'Activities':   (get_activities(gold_root),   get_activities(generated_root)),
        'Events':       (get_events(gold_root),       get_events(generated_root)),
        'Data Objects': (get_data_objects(gold_root), get_data_objects(generated_root)),
        'Conditions':   (get_conditions(gold_root),   get_conditions(generated_root)),
    }

    for category, (gold_list, gen_list) in all_elements.items():
        thresh_key = category.lower().replace(' ', '_')
        thresh = thresholds.get(thresh_key, 0.75)
        recall, matches, total, matches_list, unmatched_gold, unmatched_gen = \
            calculate_recall_for_semantic_elements(gold_list, gen_list, thresh, model)

        results[f'{category} Recall'] = recall
        results[f'{category} Absolute'] = f"{matches}/{total}"
        results[f'{category} Details'] = format_matches_for_excel(matches_list, unmatched_gold, unmatched_gen)

    gold_gateways = get_gateways_count(gold_root)
    gen_gateways  = get_gateways_count(generated_root)

    recall_and, matches_and, total_and = calculate_recall_for_counts(gold_gateways['AND'], gen_gateways['AND'])
    results['AND Gateways Recall']    = recall_and
    results['AND Gateways Absolute']  = f"{matches_and}/{total_and}"
    results['AND Gateways Details']   = f"Gold: {gold_gateways['AND']}, Generated: {gen_gateways['AND']}"

    recall_xor, matches_xor, total_xor = calculate_recall_for_counts(gold_gateways['XOR'], gen_gateways['XOR'])
    results['XOR Gateways Recall']    = recall_xor
    results['XOR Gateways Absolute']  = f"{matches_xor}/{total_xor}"
    results['XOR Gateways Details']   = f"Gold: {gold_gateways['XOR']}, Generated: {gen_gateways['XOR']}"

    return results


# ── Run ───────────────────────────────────────────────────────────────────────

base_dir = "/content"
gold_standard_dir = os.path.join(base_dir, "gold_standard")
generated_runs_folders = [
    "raw_run1", "raw_run2", "raw_run3",
    "preprocessed_run1", "preprocessed_run2", "preprocessed_run3"
]
output_excel_file = "bpmn_completeness_report.xlsx"

similarity_thresholds = {
    'actors': 0.70, 'activities': 0.50, 'events': 0.55,
    'data_objects': 0.60, 'conditions': 0.60
}

# 1. Dynamically build the list of files to compare
comparison_jobs = []

if not os.path.exists(gold_standard_dir):
    print(f"ERROR: Gold standard directory not found at {gold_standard_dir}")
else:
    for filename in os.listdir(gold_standard_dir):
        if filename.endswith(".bpmn"):
            gold_file_path = os.path.join(gold_standard_dir, filename)

            for run_folder in generated_runs_folders:
                generated_file_path = os.path.join(base_dir, run_folder, filename)

                if os.path.exists(generated_file_path):
                    comparison_jobs.append((gold_file_path, generated_file_path, run_folder))
                else:
                    print(f"Warning: File '{filename}' not found in '{run_folder}'. Skipping this pair.")

print(f"Found {len(comparison_jobs)} valid file pairs to compare.")

if comparison_jobs:
    print("Loading semantic similarity model (this may take a moment)...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Model loaded.")

    all_results = []
    print("\nStarting batch processing...")
    for gold_file, generated_file, run_folder in comparison_jobs:
        print(f"  - Comparing '{os.path.basename(gold_file)}' against '{run_folder}'")
        result_data = run_completeness_check(gold_file, generated_file, run_folder, similarity_thresholds, model)
        if result_data:
            all_results.append(result_data)

    if all_results:
        df = pd.DataFrame(all_results)

        # Added 'Run Folder' to track which folder the generated model came from
        column_order = [
            'Gold Standard', 'Run Folder', 'Generated Model',
            'Actors Recall', 'Actors Absolute',
            'Activities Recall', 'Activities Absolute',
            'Events Recall', 'Events Absolute',
            'Data Objects Recall', 'Data Objects Absolute',
            'Conditions Recall', 'Conditions Absolute',
            'AND Gateways Recall', 'AND Gateways Absolute',
            'XOR Gateways Recall', 'XOR Gateways Absolute',
            'Actors Details', 'Activities Details', 'Events Details',
            'Data Objects Details', 'Conditions Details',
            'AND Gateways Details', 'XOR Gateways Details'
        ]

        # Ensure all columns exist to prevent KeyError if some are missing
        existing_columns = [col for col in column_order if col in df.columns]
        df = df.reindex(columns=existing_columns)

        try:
            df.to_excel(output_excel_file, index=False, engine='openpyxl')
            print(f"\nSuccessfully exported {len(all_results)} results to '{output_excel_file}'")
        except Exception as e:
            print(f"\nError exporting to Excel: {e}")
    else:
        print("\nNo results to export.")
else:
    print("\nNo file pairs were found. Please check your directory structure.")

Found 60 valid file pairs to compare.
Loading semantic similarity model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.

Starting batch processing...
  - Comparing '6_eIDAS.bpmn' against 'raw_run1'
  - Comparing '6_eIDAS.bpmn' against 'raw_run2'
  - Comparing '6_eIDAS.bpmn' against 'raw_run3'
  - Comparing '6_eIDAS.bpmn' against 'preprocessed_run1'
  - Comparing '6_eIDAS.bpmn' against 'preprocessed_run2'
  - Comparing '6_eIDAS.bpmn' against 'preprocessed_run3'
  - Comparing '10_Cybersecurity.bpmn' against 'raw_run1'
  - Comparing '10_Cybersecurity.bpmn' against 'raw_run2'
  - Comparing '10_Cybersecurity.bpmn' against 'raw_run3'
  - Comparing '10_Cybersecurity.bpmn' against 'preprocessed_run1'
  - Comparing '10_Cybersecurity.bpmn' against 'preprocessed_run2'
  - Comparing '10_Cybersecurity.bpmn' against 'preprocessed_run3'
  - Comparing '8_medical_device.bpmn' against 'raw_run1'
  - Comparing '8_medical_device.bpmn' against 'raw_run2'
  - Comparing '8_medical_device.bpmn' against 'raw_run3'
  - Comparing '8_medical_device.bpmn' against 'preprocessed_run1'
  - Comparing '8_medical_device.bpm

### 2. Correctness

> **Manual LLM workflow for sub-check (iv):**  
> The *Conditional Logic* sub-check (iv) cannot be done with local embeddings—it requires an LLM to read the source text and judge the BPMN model.  
> For each file pair, the cell below will **print a self-contained prompt**. Copy it into any AI chat (Claude, ChatGPT, Gemini …), paste the JSON reply back into the notebook, and press Enter followed by `###END###`.

In [1]:
!pip install sentence-transformers torch pandas openpyxl

In [2]:
# ── Correctness check ─────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import os
import json

NAMESPACES = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}


class BPMNModel:
    def __init__(self, file_path):
        try:
            self.root = ET.parse(file_path).getroot()
            self.id_map = {elem.get('id'): elem for elem in self.root.iter() if elem.get('id')}
            self.parent_map = {c: p for p in self.root.iter() for c in p}
            self.elements_by_name = self._get_all_elements_by_name()
            self.element_to_lane_name_map = self._map_elements_to_lanes()
            self.process_to_participant_name_map = self._map_processes_to_participants()
        except ET.ParseError as e:
            raise IOError(f"Failed to parse XML file: {file_path}. Error: {e}")

    def _get_all_elements_by_name(self):
        mapping = defaultdict(list)
        for elem in self.id_map.values():
            name = elem.get('name')
            if name:
                mapping[name].append(elem)
        return mapping

    def _map_elements_to_lanes(self):
        mapping = {}
        for lane in self.root.findall('.//bpmn:lane', NAMESPACES):
            lane_name = lane.get('name')
            if lane_name:
                for node_ref in lane.findall('.//bpmn:flowNodeRef', NAMESPACES):
                    if node_ref.text:
                        mapping[node_ref.text] = lane_name
        return mapping

    def _map_processes_to_participants(self):
        mapping = {}
        for participant in self.root.findall('.//bpmn:participant', NAMESPACES):
            participant_name = participant.get('name')
            process_ref = participant.get('processRef')
            if participant_name and process_ref:
                mapping[process_ref] = participant_name
        return mapping

    def get_all_activities(self):
        activity_tags = ['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask',
                         'manualTask', 'businessRuleTask', 'scriptTask', 'callActivity', 'subProcess']
        activities = []
        for tag in activity_tags:
            activities.extend(self.root.findall(f'.//bpmn:{tag}', NAMESPACES))
        return activities

    def get_activity_actor_pairs(self):
        pairs = []
        for activity in self.get_all_activities():
            activity_name = activity.get('name')
            actor_name = self.get_actor_name_for_element(activity)
            if activity_name and actor_name:
                pairs.append((activity_name, actor_name))
        return pairs

    def get_all_flow_nodes(self):
        flow_node_tags = [
            'task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask',
            'manualTask', 'businessRuleTask', 'scriptTask', 'callActivity', 'subProcess',
            'startEvent', 'endEvent', 'intermediateThrowEvent', 'intermediateCatchEvent',
            'boundaryEvent', 'parallelGateway', 'exclusiveGateway', 'inclusiveGateway',
            'eventBasedGateway', 'complexGateway'
        ]
        nodes = []
        for tag in flow_node_tags:
            nodes.extend(self.root.findall(f'.//bpmn:{tag}', NAMESPACES))
        return nodes

    def get_control_flow_pairs(self):
        pairs = []
        for node in self.get_all_flow_nodes():
            source_name = node.get('name')
            if not source_name:
                continue
            outgoing_flows = self.root.findall(
                f".//bpmn:sequenceFlow[@sourceRef='{node.get('id')}']", NAMESPACES
            )
            for flow in outgoing_flows:
                target_id = flow.get('targetRef')
                if target_id and target_id in self.id_map:
                    target_name = self.id_map[target_id].get('name')
                    if target_name:
                        pairs.append((source_name, target_name))
        return pairs

    def get_decision_splits(self):
        splits = []
        for gw in self.root.findall('.//bpmn:exclusiveGateway[@name]', NAMESPACES):
            labels = self.get_outgoing_conditional_flow_labels(gw)
            if len(labels) > 1:
                splits.append({'name': gw.get('name'), 'labels': labels})
        return splits

    def get_element_by_name(self, name):
        return self.elements_by_name.get(name, [None])[0]

    def get_actor_name_for_element(self, element):
        if element is None:
            return None
        elem_id = element.get('id')
        if elem_id in self.element_to_lane_name_map:
            return self.element_to_lane_name_map[elem_id]
        current = element
        while current is not None:
            parent = self.parent_map.get(current)
            if parent is not None and parent.tag.endswith('process'):
                process_id = parent.get('id')
                if process_id in self.process_to_participant_name_map:
                    return self.process_to_participant_name_map[process_id]
                break
            current = parent
        return None

    def get_outgoing_conditional_flow_labels(self, element):
        if element is None:
            return []
        elem_id = element.get('id')
        labels = []
        for flow in self.root.findall(f".//bpmn:sequenceFlow[@sourceRef='{elem_id}']", NAMESPACES):
            if flow.get('name'):
                labels.append(flow.get('name'))
        return sorted(labels)


def describe_model_for_llm(model: BPMNModel) -> str:
    """Creates a textual description of a BPMN model for LLM analysis."""
    description = ["BPMN Model Structure:"]
    for p in model.root.findall('.//bpmn:participant', NAMESPACES):
        p_name = p.get('name')
        description.append(f"\n- Participant (Pool): '{p_name}'")
        process_ref = p.get('processRef')
        process = model.root.find(f".//bpmn:process[@id='{process_ref}']", NAMESPACES)
        if process is not None:
            for elem in process:
                elem_name = elem.get('name')
                elem_tag = elem.tag.split('}')[-1]
                if elem_name:
                    description.append(f"  - Contains {elem_tag}: '{elem_name}'")

    description.append("\n- Sequence Flows (Connections):")
    for flow in model.root.findall('.//bpmn:sequenceFlow', NAMESPACES):
        source_id = flow.get('sourceRef')
        target_id = flow.get('targetRef')
        source_name = model.id_map.get(source_id, ET.Element("")).get('name', 'Unnamed')
        target_name = model.id_map.get(target_id, ET.Element("")).get('name', 'Unnamed')
        flow_name = flow.get('name')
        flow_desc = f"  - From '{source_name}' to '{target_name}'"
        if flow_name:
            flow_desc += f" (Condition: '{flow_name}')"
        description.append(flow_desc)

    return "\n".join(description)


# ── Local checks (no API) ─────────────────────────────────────────────────────

def check_actor_assignment(gold_model, gen_model, embed_model, thresholds):
    gold_pairs = gold_model.get_activity_actor_pairs()
    gen_pairs  = gen_model.get_activity_actor_pairs()
    total_gold_pairs = len(gold_pairs)

    if total_gold_pairs == 0:
        return "NA" if not gen_pairs else 1.0, "0/0", "Not applicable: No (activity, actor) pairs in gold standard."
    if not gen_pairs:
        return 0.0, f"0/{total_gold_pairs}", "No (activity, actor) pairs found in the generated model."

    all_activity_names = list(set([p[0] for p in gold_pairs] + [p[0] for p in gen_pairs]))
    all_actor_names    = list(set([p[1] for p in gold_pairs] + [p[1] for p in gen_pairs]))
    activity_embeddings = {n: e for n, e in zip(all_activity_names, embed_model.encode(all_activity_names))}
    actor_embeddings    = {n: e for n, e in zip(all_actor_names,    embed_model.encode(all_actor_names))}

    def get_similarity(n1, n2, emb_dict):
        if n1 not in emb_dict or n2 not in emb_dict:
            return 0.0
        return util.pytorch_cos_sim(emb_dict[n1], emb_dict[n2]).item()

    matched_gen_indices = set()
    correctly_assigned  = 0
    details = []

    for gold_activity, gold_actor in gold_pairs:
        best_pair_score = -1
        best_gen_idx    = -1
        for gen_idx, (gen_activity, gen_actor) in enumerate(gen_pairs):
            if gen_idx in matched_gen_indices:
                continue
            sim_activity = get_similarity(gold_activity, gen_activity, activity_embeddings)
            sim_actor    = get_similarity(gold_actor,    gen_actor,    actor_embeddings)
            if sim_activity >= thresholds['activities'] and sim_actor >= thresholds['actors']:
                score = (sim_activity + sim_actor) / 2.0
                if score > best_pair_score:
                    best_pair_score = score
                    best_gen_idx    = gen_idx
        if best_gen_idx != -1:
            correctly_assigned += 1
            matched_gen_indices.add(best_gen_idx)
            mp = gen_pairs[best_gen_idx]
            details.append(f"MATCH: Gold ('{gold_activity}' by '{gold_actor}') with Gen ('{mp[0]}' by '{mp[1]}') (Score: {best_pair_score:.2f})")

    unmatched_details = []
    for gold_act, gold_acr in gold_pairs:
        if not any(f"Gold ('{gold_act}' by '{gold_acr}')" in d for d in details):
            unmatched_details.append(f"MISSING: Gold pair ('{gold_act}' by '{gold_acr}') not found in generated model.")
    for idx, (gen_act, gen_acr) in enumerate(gen_pairs):
        if idx not in matched_gen_indices:
            unmatched_details.append(f"EXTRA: Generated pair ('{gen_act}' by '{gen_acr}') has no match in gold standard.")

    score = correctly_assigned / total_gold_pairs
    return score, f"{correctly_assigned}/{total_gold_pairs}", "\n".join(details + unmatched_details)


def check_control_flow(gold_model, gen_model, embed_model, threshold):
    gold_pairs = gold_model.get_control_flow_pairs()
    gen_pairs  = gen_model.get_control_flow_pairs()
    total_gold_pairs = len(gold_pairs)

    if total_gold_pairs == 0:
        return "NA" if not gen_pairs else 1.0, "0/0", "Not applicable: No control flow pairs in gold standard."
    if not gen_pairs:
        return 0.0, f"0/{total_gold_pairs}", "No control flow pairs found in the generated model."

    all_names = list(set([n for pair in gold_pairs + gen_pairs for n in pair]))
    name_to_emb = {n: e for n, e in zip(all_names, embed_model.encode(all_names))}

    def get_similarity(n1, n2):
        if n1 not in name_to_emb or n2 not in name_to_emb:
            return 0.0
        return util.pytorch_cos_sim(name_to_emb[n1], name_to_emb[n2]).item()

    matched_gen_indices = set()
    correct_pairs = 0
    details = []

    for gs_name, gt_name in gold_pairs:
        best_pair_score = -1
        best_gen_idx    = -1
        for gen_idx, (ms_name, mt_name) in enumerate(gen_pairs):
            if gen_idx in matched_gen_indices:
                continue
            sim_source = get_similarity(gs_name, ms_name)
            sim_target = get_similarity(gt_name, mt_name)
            if sim_source >= threshold and sim_target >= threshold:
                score = (sim_source + sim_target) / 2.0
                if score > best_pair_score:
                    best_pair_score = score
                    best_gen_idx    = gen_idx
        if best_gen_idx != -1:
            correct_pairs += 1
            matched_gen_indices.add(best_gen_idx)
            mp = gen_pairs[best_gen_idx]
            details.append(f"MATCH: Gold ('{gs_name}' -> '{gt_name}') with Gen ('{mp[0]}' -> '{mp[1]}') (Score: {best_pair_score:.2f})")

    unmatched_details = []
    for gs, gt in gold_pairs:
        if not any(f"Gold ('{gs}' -> '{gt}')" in d for d in details):
            unmatched_details.append(f"MISSING: Gold pair ('{gs}' -> '{gt}') not found in generated model.")
    for idx, (ms, mt) in enumerate(gen_pairs):
        if idx not in matched_gen_indices:
            unmatched_details.append(f"EXTRA: Generated pair ('{ms}' -> '{mt}') has no match in the gold standard.")

    score = correct_pairs / total_gold_pairs
    return score, f"{correct_pairs}/{total_gold_pairs}", "\n".join(details + unmatched_details)


def check_decision_logic(gold_model, gen_model, embed_model, threshold):
    gold_splits = gold_model.get_decision_splits()
    gen_splits  = gen_model.get_decision_splits()
    total_gold_splits = len(gold_splits)

    if total_gold_splits == 0:
        return "NA" if not gen_splits else 1.0, "0/0", "Not applicable: No decision splits in gold standard."
    if not gen_splits:
        return 0.0, f"0/{total_gold_splits}", "No decision splits found in the generated model."

    all_gw_names = list(set([s['name'] for s in gold_splits] + [s['name'] for s in gen_splits]))
    all_labels   = list(set(
        [lbl for s in gold_splits for lbl in s['labels']] +
        [lbl for s in gen_splits  for lbl in s['labels']]
    ))
    gw_emb    = {n: e for n, e in zip(all_gw_names, embed_model.encode(all_gw_names))}
    label_emb = {n: e for n, e in zip(all_labels,   embed_model.encode(all_labels))}

    def get_gw_sim(n1, n2):
        if n1 not in gw_emb or n2 not in gw_emb:
            return 0.0
        return util.pytorch_cos_sim(gw_emb[n1], gw_emb[n2]).item()

    def get_label_set_recall(gold_labels, gen_labels):
        if not gold_labels:
            return 1.0
        if not gen_labels:
            return 0.0
        label_matches = 0
        used_gen = set()
        for gl in gold_labels:
            best_score = -1
            best_idx   = -1
            for i, ml in enumerate(gen_labels):
                if i in used_gen:
                    continue
                score = util.pytorch_cos_sim(label_emb[gl], label_emb[ml]).item()
                if score > best_score:
                    best_score = score
                    best_idx   = i
            if best_score >= threshold:
                label_matches += 1
                if best_idx != -1:
                    used_gen.add(best_idx)
        return label_matches / len(gold_labels)

    correct_splits      = 0
    matched_gen_indices = set()
    details = []

    for gold_split in gold_splits:
        best_overall_score = -1
        best_gen_idx       = -1
        for gen_idx, gen_split in enumerate(gen_splits):
            if gen_idx in matched_gen_indices:
                continue
            gw_sim = get_gw_sim(gold_split['name'], gen_split['name'])
            if gw_sim >= threshold:
                label_recall   = get_label_set_recall(gold_split['labels'], gen_split['labels'])
                overall_score  = (gw_sim + label_recall) / 2.0
                if overall_score > best_overall_score:
                    best_overall_score = overall_score
                    best_gen_idx       = gen_idx
        if best_gen_idx != -1:
            correct_splits += 1
            matched_gen_indices.add(best_gen_idx)
            details.append(f"MATCH: Gold Split '{gold_split['name']}' with Gen Split '{gen_splits[best_gen_idx]['name']}' (Score: {best_overall_score:.2f})")

    unmatched_details = []
    for gs in gold_splits:
        if not any(f"'{gs['name']}'" in d for d in details):
            unmatched_details.append(f"MISSING: Gold split '{gs['name']}' not found in generated model.")
    for idx, gs in enumerate(gen_splits):
        if idx not in matched_gen_indices:
            unmatched_details.append(f"EXTRA: Generated split '{gs['name']}' has no match in gold standard.")

    score = correct_splits / total_gold_splits
    return score, f"{correct_splits}/{total_gold_splits}", "\n".join(details + unmatched_details)


# ── Sub-check (iv): Conditional Logic — one prompt per document ───────────────

def check_conditional_logic_for_document(
    source_text: str,
    run_models: dict,       # {run_folder: BPMNModel}
    doc_name: str = ""
) -> dict:
    """
    Builds ONE prompt for all run-folder variants of a single document,
    prints it, reads a single pasted JSON response, and returns per-run results.

    Returns:
        {run_folder: (score, absolute, details_str)}
    """
    # Build a numbered section for every model so the LLM can address each one
    model_sections = []
    run_folder_index = {}   # index (1-based) -> run_folder name
    for idx, (run_folder, gen_model) in enumerate(run_models.items(), start=1):
        run_folder_index[idx] = run_folder
        model_desc = describe_model_for_llm(gen_model)
        model_sections.append(
            f"### Model {idx} — run folder: \"{run_folder}\"\n{model_desc}"
        )

    models_block = "\n\n".join(model_sections)

    prompt = f"""\
You are an expert BPMN 2.0 analyst. Your task is to analyse a source text, identify \
all business rules within it, and then evaluate whether each rule is correctly \
implemented in EACH of the {len(run_models)} generated BPMN models listed below.

**Source Text:**
```
{source_text}
```

**Generated BPMN Models ({len(run_models)} variants of document "{doc_name}"):**

{models_block}

**Analysis Task:**
1. Read the entire Source Text and identify every sentence that constitutes a business \
rule, a constraint, or a condition (e.g. sentences containing "must", "if", "can", \
"within", etc.).
2. For EACH of the {len(run_models)} models listed above, evaluate every identified \
rule independently.
3. Consider whether conditional clauses use gateways, temporal constraints use timer \
events, and deontic logic (must, can, must not) is correctly modelled.

**Output Format:**
Respond with a single JSON object only — no preamble, no markdown fences.
The object must have one key per model index (as a string: "1", "2", … "{len(run_models)}").
Each value is a JSON array of rule-evaluation objects with three keys:
  "identified_rule"  (string),
  "evaluation"       ("Correct" or "Incorrect"),
  "justification"    (string)

Example structure (two models, two rules each):
{{
  "1": [
    {{"identified_rule": "...", "evaluation": "Correct",   "justification": "..."}},
    {{"identified_rule": "...", "evaluation": "Incorrect", "justification": "..."}}
  ],
  "2": [
    {{"identified_rule": "...", "evaluation": "Incorrect", "justification": "..."}},
    {{"identified_rule": "...", "evaluation": "Correct",   "justification": "..."}}
  ]
}}"""

    # ── Print prompt and collect response ─────────────────────────────────────
    separator = "=" * 80
    print(f"\n{separator}")
    print(f"DOCUMENT: {doc_name}  |  {len(run_models)} models to evaluate")
    print(separator)
    print(prompt)
    print(f"\n{separator}")
    print("Paste the LLM's JSON response below, then press Enter twice + Ctrl-D (or Ctrl-Z on Windows):")
    print(separator)

    lines = []
    try:
        while True:
            lines.append(input())
    except EOFError:
        pass
    response_text = "\n".join(lines)

    # ── Parse and distribute results ──────────────────────────────────────────
    per_run_results: dict[str, tuple] = {}

    try:
        clean = response_text.strip().replace("```json", "").replace("```", "")
        parsed = json.loads(clean)

        if not isinstance(parsed, dict):
            raise ValueError("Top-level JSON value is not an object.")

        for idx, run_folder in run_folder_index.items():
            key = str(idx)
            eval_list = parsed.get(key)

            if eval_list is None:
                per_run_results[run_folder] = (
                    "NA", "0/0",
                    f"No entry for model index '{key}' found in the LLM response."
                )
                continue

            if not isinstance(eval_list, list):
                per_run_results[run_folder] = (
                    "NA", "0/0",
                    f"Entry for model index '{key}' is not a list."
                )
                continue

            correct = 0
            details = []
            for item in eval_list:
                rule          = item.get("identified_rule", "N/A")
                evaluation    = item.get("evaluation", "Error")
                justification = item.get("justification", "No justification provided.")
                if evaluation.strip().lower() == "correct":
                    correct += 1
                details.append(
                    f"Rule: '{rule}' -> {evaluation}. Justification: {justification}"
                )

            total = len(eval_list)
            score = correct / total if total > 0 else "NA"
            per_run_results[run_folder] = (score, f"{correct}/{total}", "\n".join(details))

    except (json.JSONDecodeError, ValueError, AttributeError) as e:
        error_msg = f"Error parsing LLM response: {e}\nRaw response:\n{response_text}"
        for run_folder in run_models:
            per_run_results[run_folder] = ("NA", "0/0", error_msg)

    return per_run_results


def run_correctness_check(gold_file, gen_file, run_folder, thresholds, embed_model,
                           source_text, llm_result: tuple):
    """
    Runs sub-checks (i)–(iii) locally and accepts the pre-computed (iv) result
    from the shared per-document LLM call.
    """
    results = {
        'Gold Standard':   os.path.basename(gold_file),
        'Run Folder':      run_folder,
        'Generated Model': os.path.basename(gen_file),
    }

    try:
        gold_model = BPMNModel(gold_file)
        gen_model  = BPMNModel(gen_file)
    except IOError as e:
        print(f"  - SKIPPING: Could not process file. Error: {e}")
        return None

    score, absolute, details = check_actor_assignment(gold_model, gen_model, embed_model, thresholds)
    results['(i) Actor Assignment Score']    = score
    results['(i) Actor Assignment Absolute'] = absolute
    results['(i) Actor Assignment Details']  = details

    score, absolute, details = check_control_flow(gold_model, gen_model, embed_model, thresholds['activities'])
    results['(ii) Control Flow Score']    = score
    results['(ii) Control Flow Absolute'] = absolute
    results['(ii) Control Flow Details']  = details

    score, absolute, details = check_decision_logic(gold_model, gen_model, embed_model, thresholds['conditions'])
    results['(iii) Decision Logic Score']    = score
    results['(iii) Decision Logic Absolute'] = absolute
    results['(iii) Decision Logic Details']  = details

    # (iv) comes from the shared per-document LLM call
    score, absolute, details = llm_result
    results['(iv) Conditional Logic (LLM) Score']    = score
    results['(iv) Conditional Logic (LLM) Absolute'] = absolute
    results['(iv) Conditional Logic (LLM) Details']  = details

    return results


# ── Run ───────────────────────────────────────────────────────────────────────

base_dir = "/content"
gold_standard_dir = os.path.join(base_dir, "gold_standard")
source_text_dir   = os.path.join(base_dir, "text")
generated_runs_folders = [
    "raw_run1", "raw_run2", "raw_run3",
    "preprocessed_run1", "preprocessed_run2", "preprocessed_run3"
]

output_excel_file       = "bpmn_correctness_report.xlsx"
similarity_thresholds   = {'activities': 0.50, 'conditions': 0.50, 'actors': 0.70}

# ── 1. Group comparisons by document (gold filename) ─────────────────────────
# Structure: {filename: {"gold": path, "source_text": path, "runs": {run_folder: gen_path}}}
documents: dict[str, dict] = {}

if not os.path.exists(gold_standard_dir):
    print(f"ERROR: Gold standard directory not found at {gold_standard_dir}")
elif not os.path.exists(source_text_dir):
    print(f"ERROR: Source text directory not found at {source_text_dir}")
else:
    for filename in os.listdir(gold_standard_dir):
        if not filename.endswith(".bpmn"):
            continue

        gold_file_path = os.path.join(gold_standard_dir, filename)

        doc_prefix = filename.split("_")[0] + "_"
        source_text_file = next(
            (os.path.join(source_text_dir, f)
             for f in os.listdir(source_text_dir)
             if f.startswith(doc_prefix) and f.endswith(".txt")),
            None
        )
        if not source_text_file:
            print(f"Warning: No source text for '{filename}' (prefix '{doc_prefix}'). Skipping.")
            continue

        run_paths = {}
        for run_folder in generated_runs_folders:
            gen_path = os.path.join(base_dir, run_folder, filename)
            if os.path.exists(gen_path):
                run_paths[run_folder] = gen_path
            else:
                print(f"Warning: '{filename}' not found in '{run_folder}'. Skipping that run.")

        if run_paths:
            documents[filename] = {
                "gold":        gold_file_path,
                "source_text": source_text_file,
                "runs":        run_paths,
            }

print(f"Found {len(documents)} documents with at least one generated run.\n")

if documents:
    print("Loading semantic similarity model...")
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Model loaded.\n")

    all_results = []

    for filename, doc_info in documents.items():
        gold_file        = doc_info["gold"]
        source_text_file = doc_info["source_text"]
        run_paths        = doc_info["runs"]       # {run_folder: gen_path}

        try:
            with open(source_text_file, 'r', encoding='utf-8') as f:
                source_text = f.read()
        except Exception as e:
            print(f"  - SKIPPING '{filename}': cannot read source text. Error: {e}")
            continue

        # ── Load all generated models for this document ───────────────────────
        run_models: dict[str, BPMNModel] = {}
        for run_folder, gen_path in run_paths.items():
            try:
                run_models[run_folder] = BPMNModel(gen_path)
            except IOError as e:
                print(f"  - WARNING: Could not load '{run_folder}/{filename}': {e}")

        if not run_models:
            print(f"  - SKIPPING '{filename}': no generated models could be loaded.")
            continue

        # ── (iv) ONE prompt for all runs of this document ─────────────────────
        print(f"\nDocument '{filename}': sending ONE combined prompt for {len(run_models)} run(s).")
        llm_results_per_run = check_conditional_logic_for_document(
            source_text=source_text,
            run_models=run_models,
            doc_name=filename,
        )

        # ── (i)–(iii) per run, (iv) from the shared result ───────────────────
        for run_folder, gen_model in run_models.items():
            gen_path   = run_paths[run_folder]
            llm_result = llm_results_per_run.get(
                run_folder,
                ("NA", "0/0", "No LLM result returned for this run.")
            )

            print(f"  - Running local checks: '{filename}' vs '{run_folder}'")
            result_data = run_correctness_check(
                gold_file     = gold_file,
                gen_file      = gen_path,
                run_folder    = run_folder,
                thresholds    = similarity_thresholds,
                embed_model   = embed_model,
                source_text   = source_text,
                llm_result    = llm_result,
            )
            if result_data:
                all_results.append(result_data)

    if all_results:
        df = pd.DataFrame(all_results)
        column_order = [
            'Gold Standard', 'Run Folder', 'Generated Model',
            '(i) Actor Assignment Score',         '(i) Actor Assignment Absolute',
            '(ii) Control Flow Score',            '(ii) Control Flow Absolute',
            '(iii) Decision Logic Score',         '(iii) Decision Logic Absolute',
            '(iv) Conditional Logic (LLM) Score', '(iv) Conditional Logic (LLM) Absolute',
            '(i) Actor Assignment Details',       '(ii) Control Flow Details',
            '(iii) Decision Logic Details',       '(iv) Conditional Logic (LLM) Details',
        ]
        existing_columns = [c for c in column_order if c in df.columns]
        df = df.reindex(columns=existing_columns)

        try:
            df.to_excel(output_excel_file, index=False, engine='openpyxl')
            print(f"\nSuccessfully exported {len(all_results)} results to '{output_excel_file}'")
        except Exception as e:
            print(f"\nError exporting to Excel: {e}")
    else:
        print("\nNo results to export.")
else:
    print("\nNo documents found. Please check your directory structure.")

Found 10 documents with at least one generated run.

Loading semantic similarity model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
  - From 'communicate to the Commission and the Board the names...' to ''
  - From 'assess whether it is following award... or following suspension...' to '' (Condition: 'no')
  - From '' to 'Receive reports from the Trusted Flagger'
  - From 'Receive reports from the Trusted Flagger' to 'Receive information from an Online Platform Provider'
  - From 'Receive information from an Online Platform Provider' to 'assess whether legitimate reasons to open an investigation'
  - From 'assess whether legitimate reasons to open an investigation' to 'suspend the status of trusted flagger' (Condition: 'yes')
  - From 'suspend the status of trusted flagger' to 'carry out the investigation without undue delay'
  - From 'carry out the investigation without undue delay' to 'afford the entity an opportunity to react to the investigation findings'
  - From 'afford the entity an opportunity to react to the investigation findings' to ''
  

### 3. Traceability

In [1]:
!pip install sentence-transformers torch pandas openpyxl

In [3]:
import xml.etree.ElementTree as ET
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import os
import json

NAMESPACES = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}

# ── Shared BPMN Logic ────────────────────────────────────────────────────────

class BPMNModel:
    def __init__(self, file_path):
        try:
            self.root = ET.parse(file_path).getroot()
            self.id_map = {elem.get('id'): elem for elem in self.root.iter() if elem.get('id')}
        except ET.ParseError as e:
            raise IOError(f"Failed to parse XML file: {file_path}. Error: {e}")

    def get_elements_by_tag(self, tag_name):
        return self.root.findall(f'.//bpmn:{tag_name}', NAMESPACES)

    def get_element_names(self, elements):
        names = [elem.get('name') for elem in elements]
        return sorted(list(set([name for name in names if name])))

    def get_actors(self):
        participants = self.get_elements_by_tag('participant')
        lanes = self.get_elements_by_tag('lane')
        return self.get_element_names(participants + lanes)

    def get_activities(self):
        activity_tags = ['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask',
                         'manualTask', 'businessRuleTask', 'scriptTask', 'callActivity', 'subProcess']
        activities = []
        for tag in activity_tags:
            activities.extend(self.get_elements_by_tag(tag))
        return self.get_element_names(activities)

def describe_elements_for_llm(model: BPMNModel) -> str:
    actors = model.get_actors()
    activities = model.get_activities()
    desc = "Actors: " + (", ".join([f"'{a}'" for a in actors]) if actors else "None")
    desc += "\nActivities: " + (", ".join([f"'{a}'" for a in activities]) if activities else "None")
    return desc

# ── Traceability Logic (LLM Integration) ──────────────────────────────────────

def check_traceability_with_llm(source_text, run_models, doc_name):
    """ Builds ONE prompt to check if model elements are traceable to the text. """
    model_sections = []
    run_folder_index = {}
    for idx, (run_folder, gen_model) in enumerate(run_models.items(), start=1):
        run_folder_index[idx] = run_folder
        model_sections.append(f"### Model {idx} (Run: {run_folder})\n{describe_elements_for_llm(gen_model)}")

    models_block = "\n\n".join(model_sections)

    prompt = f"""
You are a BPMN traceability expert. Evaluate if the Actors and Activities in the models are traceable to the source text.

**Source Text:**
{source_text}

**Models to Evaluate:**
{models_block}

**Task:**
For each model, determine if the extracted elements (Actors/Activities) accurately represent the entities and actions in the text.
Identify:
1. Missing elements (required by text but not in model).
2. Hallucinated elements (in model but not justified by text).

**Output Format:**
JSON only. Keyed by model index.
{{
  "1": {{"score": 0.85, "details": "Matches well, but missing actor X."}},
  ...
}}
"""
    print(f"\n{'='*30}\nDOCUMENT: {doc_name}\n{'='*30}\n{prompt}\n{'='*30}")
    print("Paste JSON response then Enter twice + Ctrl-D:")

    lines = []
    try:
        while True:
            line = input()
            if not line and lines and not lines[-1]: break
            lines.append(line)
    except EOFError: pass

    response_text = "\n".join(lines)
    per_run_results = {}
    try:
        clean = response_text.strip().replace("```json", "").replace("```", "")
        parsed = json.loads(clean)
        for idx, run_folder in run_folder_index.items():
            res = parsed.get(str(idx), {"score": 0, "details": "No data"})
            per_run_results[run_folder] = (res.get("score"), res.get("details"))
    except:
        for run_folder in run_models: per_run_results[run_folder] = ("NA", "Error parsing LLM")

    return per_run_results

# ── Main Execution ───────────────────────────────────────────────────────────

base_dir = "/content"
gold_standard_dir = os.path.join(base_dir, "gold_standard")
source_text_dir   = os.path.join(base_dir, "text")
generated_runs_folders = ["raw_run1", "raw_run2", "raw_run3", "preprocessed_run1", "preprocessed_run2", "preprocessed_run3"]

documents = {}
# (Logic to populate documents dict identical to your Correctness Check)
for filename in os.listdir(gold_standard_dir):
    if filename.endswith(".bpmn"):
        doc_prefix = filename.split("_")[0] + "_"
        source_path = next((os.path.join(source_text_dir, f) for f in os.listdir(source_text_dir) if f.startswith(doc_prefix)), None)
        if source_path:
            runs = {rf: os.path.join(base_dir, rf, filename) for rf in generated_runs_folders if os.path.exists(os.path.join(base_dir, rf, filename))}
            documents[filename] = {"gold": os.path.join(gold_standard_dir, filename), "source": source_path, "runs": runs}

all_results = []
for filename, info in documents.items():
    with open(info["source"], 'r') as f: source_text = f.read()

    run_models = {rf: BPMNModel(path) for rf, path in info["runs"].items()}
    llm_results = check_traceability_with_llm(source_text, run_models, filename)

    for run_folder, model in run_models.items():
        score, details = llm_results.get(run_folder, ("NA", "N/A"))
        all_results.append({
            'Gold Standard': filename,
            'Run Folder': run_folder,
            'Actors': ", ".join(model.get_actors()),
            'Activities': ", ".join(model.get_activities()),
            'Traceability Score': score,
            'LLM Feedback': details
        })

df = pd.DataFrame(all_results)
df.to_excel("bpmn_traceability_report.xlsx", index=False)
print("Done.")


DOCUMENT: 4_health_care.bpmn

You are a BPMN traceability expert. Evaluate if the Actors and Activities in the models are traceable to the source text.

**Source Text:**
Exchange of health data before a transfer is carried out
1. For the sole purpose of the provision of medical care or treatment, in particular concerning disabled persons, elderly people, pregnant women, minors and persons who have been subject to torture, rape or other serious forms of psychological, physical and sexual violence, the transferring Member State shall, in so far as it is available to the competent authority in accordance with national law, transmit to the Member State responsible information on any special needs of the person to be transferred, which in specific cases may include information on that person’s physical or mental health. That information shall be transferred in a common health certificate with the necessary documents attached. The Member State responsible shall ensure that those special nee

In [ ]:
# ── Traceability check (no API needed) ───────────────────────────────────────
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import os

NAMESPACES = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}


def get_elements_by_tag(root, tag_name):
    return root.findall(f'.//bpmn:{tag_name}', NAMESPACES)

def get_element_names(elements):
    names = [elem.get('name') for elem in elements]
    return [name for name in names if name]

def get_actors(root):
    participants = get_elements_by_tag(root, 'participant')
    lanes = get_elements_by_tag(root, 'lane')
    return get_element_names(participants + lanes)

def get_activities(root):
    activity_tags = ['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask',
                     'manualTask', 'businessRuleTask', 'scriptTask', 'callActivity', 'subProcess']
    activities = []
    for tag in activity_tags:
        activities.extend(get_elements_by_tag(root, tag))
    return get_element_names(activities)


def calculate_precision_for_semantic_elements(gold_standard_names, generated_names, similarity_threshold, model):
    if not generated_names:
        return 1.0, 0, 0, [], gold_standard_names, []
    if not gold_standard_names:
        return 0.0, 0, len(generated_names), [], [], generated_names

    gold_embeddings      = model.encode(gold_standard_names, convert_to_tensor=True)
    generated_embeddings = model.encode(generated_names,     convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(gold_embeddings, generated_embeddings)

    matches_list = []
    unmatched_gold_indices      = set(range(len(gold_standard_names)))
    unmatched_generated_indices = set(range(len(generated_names)))
    matched_generated_indices   = set()

    for gs_idx in range(len(gold_standard_names)):
        best_score  = -1
        best_gen_idx = -1
        for gen_idx in range(len(generated_names)):
            if gen_idx in matched_generated_indices:
                continue
            score = cosine_scores[gs_idx][gen_idx]
            if score > best_score:
                best_score   = score
                best_gen_idx = gen_idx

        if best_score >= similarity_threshold:
            matches_list.append(
                (gold_standard_names[gs_idx], generated_names[best_gen_idx], best_score.item())
            )
            if best_gen_idx != -1:
                matched_generated_indices.add(best_gen_idx)
            if gs_idx in unmatched_gold_indices:
                unmatched_gold_indices.remove(gs_idx)
            if best_gen_idx in unmatched_generated_indices:
                unmatched_generated_indices.remove(best_gen_idx)

    matches_count    = len(matches_list)
    total_generated  = len(generated_names)
    precision        = matches_count / total_generated if total_generated > 0 else 1.0
    unmatched_gold   = [gold_standard_names[i] for i in unmatched_gold_indices]
    unmatched_gen    = [generated_names[i]     for i in unmatched_generated_indices]

    return precision, matches_count, total_generated, matches_list, unmatched_gold, unmatched_gen


def format_matches_for_excel(matches_list, unmatched_gold, unmatched_gen):
    report_parts = []
    if matches_list:
        report_parts.append("MATCHED (Correctly Generated):")
        for gs, gen, score in sorted(matches_list, key=lambda x: x[2], reverse=True):
            report_parts.append(f"  - G: '{gs}' <-> M: '{gen}' ({score:.2f})")
    if unmatched_gen:
        report_parts.append("\nEXTRA IN MODEL (Incorrectly Generated):")
        for name in sorted(unmatched_gen):
            report_parts.append(f"  - {name}")
    if unmatched_gold:
        report_parts.append("\nMISSING FROM MODEL (Not relevant for precision but good to know):")
        for name in sorted(unmatched_gold):
            report_parts.append(f"  - {name}")
    return "\n".join(report_parts)


def run_traceability_check(gold_standard_file, generated_file, thresholds, model):
    results = {
        'Gold Standard':   os.path.basename(gold_standard_file),
        'Generated Model': os.path.basename(generated_file),
    }

    try:
        gold_root = ET.parse(gold_standard_file).getroot()
    except (FileNotFoundError, ET.ParseError) as e:
        print(f"Error reading gold standard file '{gold_standard_file}': {e}")
        return None

    try:
        generated_root = ET.parse(generated_file).getroot()
    except (FileNotFoundError, ET.ParseError) as e:
        print(f"Error reading generated file '{generated_file}': {e}")
        return None

    traceability_elements = {
        'Actors':     (get_actors(gold_root),     get_actors(generated_root)),
        'Activities': (get_activities(gold_root), get_activities(generated_root)),
    }

    for category, (gold_list, gen_list) in traceability_elements.items():
        thresh_key = category.lower().replace(' ', '_')
        thresh = thresholds.get(thresh_key, 0.75)
        precision, matches, total_generated, matches_list, unmatched_gold, unmatched_gen = \
            calculate_precision_for_semantic_elements(gold_list, gen_list, thresh, model)

        results[f'{category} Precision']          = precision
        results[f'{category} Absolute (Precision)'] = f"{matches}/{total_generated}"
        results[f'{category} Details']            = format_matches_for_excel(matches_list, unmatched_gold, unmatched_gen)

    return results


# ── Run ───────────────────────────────────────────────────────────────────────
comparison_files = [
    ("/content/smart_meter_refactored.bpmn",         "/content/smart_meter_01.bpmn"),
    ("/content/smart_meter_refactored.bpmn",         "/content/smart_meter_02.bpmn"),
    ("/content/smart_meter_refactored.bpmn",         "/content/smart_meter_03.bpmn"),
    ("/content/gdpr_refactored.bpmn",                "/content/gdpr_01.bpmn"),
    ("/content/gdpr_refactored.bpmn",                "/content/gdpr_02.bpmn"),
    ("/content/gdpr_refactored.bpmn",                "/content/gdpr_03.bpmn"),
    ("/content/blood_donor_selection.bpmn",          "/content/blood_donor_01.bpmn"),
    ("/content/blood_donor_selection.bpmn",          "/content/blood_donor_02.bpmn"),
    ("/content/blood_donor_selection.bpmn",          "/content/blood_donor_03.bpmn"),
    ("/content/health_data.bpmn",                    "/content/health_data_01.bpmn"),
    ("/content/health_data.bpmn",                    "/content/health_data_02.bpmn"),
    ("/content/health_data.bpmn",                    "/content/health_data_03.bpmn"),
    ("/content/finance_customer_due_diligence.bpmn", "/content/CDD_01.bpmn"),
    ("/content/finance_customer_due_diligence.bpmn", "/content/CDD_02.bpmn"),
    ("/content/finance_customer_due_diligence.bpmn", "/content/CDD_03.bpmn"),
]

output_excel_file = "bpmn_traceability_report_CARB.xlsx"

similarity_thresholds = {'actors': 0.70, 'activities': 0.5}

print("Loading semantic similarity model (this may take a moment)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

all_results = []
print("\nStarting traceability (precision) check...")
for gold_file, generated_file in comparison_files:
    print(f"  - Comparing '{os.path.basename(gold_file)}' with '{os.path.basename(generated_file)}'")
    result_data = run_traceability_check(gold_file, generated_file, similarity_thresholds, model)
    if result_data:
        all_results.append(result_data)

if all_results:
    df = pd.DataFrame(all_results)
    column_order = [
        'Gold Standard', 'Generated Model',
        'Actors Precision', 'Actors Absolute (Precision)',
        'Activities Precision', 'Activities Absolute (Precision)',
        'Actors Details', 'Activities Details'
    ]
    df = df.reindex(columns=column_order)
    try:
        df.to_excel(output_excel_file, index=False, engine='openpyxl')
        print(f"\nSuccessfully exported {len(all_results)} results to '{output_excel_file}'")
    except Exception as e:
        print(f"\nError exporting to Excel: {e}")
else:
    print("\nNo results to export.")

---
# III. Layout Eval

> **Manual LLM workflow:**  
> For each image, the cell below will print a prompt **and tell you exactly which image file to upload** in your AI chat. Paste the JSON response back and type `###END###` to continue.

In [ ]:
!pip install pandas openpyxl

In [ ]:
# ── Layout Eval ───────────────────────────────────────────────────────────────
import pandas as pd
import os
import json


def run_layout_check(image_path: str) -> dict:
    """
    Replaces the Gemini Vision API call.

    Prints instructions + prompt so you can upload the image in any
    vision-capable AI chat, then collects the JSON response interactively.
    """
    results = {
        'Model Image':   os.path.basename(image_path),
        'Overall Score': None,
        'Issues Count':  0,
        'Issues Details': 'N/A'
    }

    if not os.path.exists(image_path):
        print(f"  - WARNING: Image file not found: {image_path}")
        results['Issues Details'] = "Error: Image file not found."
        return results

    response_text = manual_llm_layout_query(image_path)

    try:
        clean_response = response_text.strip().replace("```json", "").replace("```", "")
        llm_eval = json.loads(clean_response)

        if "error" in llm_eval:
            raise ValueError(llm_eval["error"])

        results['Overall Score'] = llm_eval.get('overall_score', 0.0)
        issues = llm_eval.get('identified_issues', [])
        results['Issues Count'] = len(issues)

        if issues:
            details_list = [
                f"- {item.get('issue_description', 'N/A')} (Severity: {item.get('severity', 'N/A')})"
                for item in issues
            ]
            results['Issues Details'] = "\n".join(details_list)
        else:
            results['Issues Details'] = "No layout issues identified."

    except (json.JSONDecodeError, AttributeError, ValueError) as e:
        results['Issues Details'] = f"Error parsing LLM response: {e}\nRaw Response: {response_text}"

    return results


# ── Run ───────────────────────────────────────────────────────────────────────
image_files_to_check = [
    "/content/smart_meter_01.png",
    "/content/gdpr_01.png",
    "/content/blood_donor_01.png",
    "/content/health_data_01.png",
    "/content/CDD_01.png",
]

output_excel_file = "bpmn_layout_report_CARB.xlsx"

all_results = []
print("\nStarting BPMN layout analysis...")
print("You will be prompted once per image to paste the AI's JSON response.\n")

for image_file in image_files_to_check:
    print(f"  - Analyzing '{os.path.basename(image_file)}'...")
    result_data = run_layout_check(image_file)
    if result_data:
        all_results.append(result_data)

if all_results:
    df = pd.DataFrame(all_results)
    column_order = ['Model Image', 'Overall Score', 'Issues Count', 'Issues Details']
    df = df.reindex(columns=column_order)
    try:
        df.to_excel(output_excel_file, index=False, engine='openpyxl')
        print(f"\nSuccessfully exported {len(all_results)} results to '{output_excel_file}'")
    except Exception as e:
        print(f"\nError exporting to Excel: {e}")
else:
    print("\nNo results to export.")